In [0]:
# %pip install -U -qqqq 
# backoff 
# databricks-langchain 
# langgraph==0.5.3 
# uv 
# databricks-agents 
# mlflow-skinny[databricks] 
# chromadb 
# sentence-transformers 
# langchain-huggingface
# langchain-chroma 
# wikipedia 
# faiss-cpu

In [0]:
%pip install -U -q databricks-langchain langchain==0.3.7 faiss-cpu wikipedia langchain-community chromadb langchain-openai tiktoken


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from databricks_langchain import ChatDatabricks, DatabricksEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import RetrievalQA

### Simple RAG

In [0]:
# Retriever Config
MAX_WIKI_DOCS_PER_TOPIC = 10 #TODO: recommend starting with a smaller number for testing purposes
VECTOR_TOP_K = 5 # number of documents to return
EMBEDDING_MODEL = "databricks-bge-large-en" # Embedding model endpoint name

# LLM Config
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-1-8b-instruct"

# Initialize embeddings + LLM
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.2)


In [0]:
from langchain.document_loaders import WikipediaLoader

loader = WikipediaLoader(query="deep learning", load_max_docs=5)
docs = loader.load()

In [0]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)
print(f"len chunk {len(chunks)} ")

len chunk 68 


In [0]:
from langchain_community.vectorstores import FAISS

embeddings = DatabricksEmbeddings(endpoint=EMBEDDING_MODEL)

# Build FAISS index from your chunks
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [0]:
# Initialize embeddings + LLM
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.2)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # 检索Top-3相关片段

In [0]:
prompt = ChatPromptTemplate.from_template("""
Please answer the question based on the following context information.
If the context does not provide relevant information, please directly say:
"Based on the available information, I cannot answer this question."

Context: {context}

Question: {question}

Answer:
""")

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # The simplest chain type: stuff all retrieved context into the prompt
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

# 4. Test query
query = "what is deep learning?"
result = rag_chain.invoke({"query": query})

print(f"Question: {query}")
print(f"Answer: {result['result']}")

Question: what is deep learning?
Answer: Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network.


### Conversation RAG

In [0]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

# 1. Add a memory module on top of standard RAG
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 2. Build a conversational RAG chain
conversational_rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": prompt}  # You can use a more complex prompt if needed
)

# 3. Simulate a multi-turn conversation
print("--- First Round of Conversation ---")
result1 = conversational_rag_chain.invoke({"question": "What is deep learning?"})
print(f"User: What is deep learning")
print(f"AI: {result1['answer']}")

print("\n--- Second Round of Conversation (Context-Dependent) ---")
result2 = conversational_rag_chain.invoke({"question": "How to learn deep learning?"})
print(f"User: How to learn deep learning?")
print(f"AI: {result2['answer']}")

/home/spark-06f991d3-cfde-4664-a382-e7/.ipykernel/1875707/command-5881766325413111-95536204:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


--- First Round of Conversation ---
User: What is deep learning
AI: Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network, using a hierarchy of layers to transform input data into a progressively more abstract and composite representation.

--- Second Round of Conversation (Context-Dependent) ---
User: How to learn deep learning?
AI: Based on the available information, I cannot answer this question.

However, I can provide some general information on how to learn deep learning. Deep learning is a complex and rapidly evolving field, and there are many resources available to learn it. Here are some general steps:

1. **Get familiar with machine learning basics**: Understand the fundamentals of machine learning, including supervised and unsupervised learning, regression, classification, and neural networks.
2. **Choose a programming language**: Python is a popular choice for deep learning, and libraries like Tens

### Corrective RAG, CRAG

工作原理：

检索阶段： 从内部向量存储库获取文档。
评估阶段： 轻量级“评分器”模型为每个文档片段赋予评分（正确/模糊/错误）。
决策门控：
- 正确： 直接进入生成器阶段
- 错误： 丢弃数据并触发外部API（如谷歌搜索或Tavily）
- 合成： 基于验证过的内部数据或新获取的外部数据生成答案

In [0]:
# Conceptual code illustrating the decision logic of CRAG (Corrective RAG)
def corrective_rag_workflow(query, vectorstore, web_search_tool):
    # 1. Initial retrieval
    retrieved_docs = vectorstore.similarity_search(query, k=5)
    
    # 2. Evaluate retrieved results (simulate a lightweight evaluator)
    # In practice, this could be a small trained model or a rule-based scorer
    graded_docs = []
    for doc in retrieved_docs:
        # Simple simulation: score by checking keyword overlap with the query
        relevance_score = naive_relevance_scorer(query, doc.page_content)
        if relevance_score > 0.7:
            graded_docs.append(("correct", doc))
        elif relevance_score > 0.3:
            graded_docs.append(("ambiguous", doc))
        else:
            graded_docs.append(("incorrect", doc))
    
    # 3. Decision gate
    if any(grade == "correct" for grade, _ in graded_docs):
        # If we have qualified documents, use them
        context = "\n".join([d.page_content for g, d in graded_docs if g == "correct"])
        print("[CRAG Decision]: Using internal knowledge base.")
    else:
        # If internal docs are low quality, trigger fallback retrieval
        print("[CRAG Decision]: Insufficient internal knowledge, starting fallback retrieval.")
        context = web_search_tool.search(query)
    
    # 4. Generation
    final_prompt = f"Answer the question based on the following information:\n{context}\n\nQuestion: {query}\nAnswer:"
    return llm.invoke(final_prompt)

# A simplistic relevance scorer
def naive_relevance_scorer(query, doc_content):
    query_words = set(query.lower().split())
    doc_words = set(doc_content.lower().split())
    overlap = len(query_words & doc_words)
    return overlap / max(len(query_words), 1)

# Assume we have a web-connected search tool (e.g., a wrapper around the Tavily Search API)
# web_search_tool = TavilySearch()

In [0]:
class MockWebSearchTool:
    def search(self, query):
        if "deep learning" in query.lower():
            return (
                "Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network, using a hierarchy of layers to transform input data into a progressively more abstract and composite representation."
            )
        elif "latest breakthrough" in query.lower():
            return (
                "Recent breakthroughs in deep learning include improvements in "
                "large language models, multimodal systems, and efficient fine-tuning methods."
            )
        else:
            return f"External search result related to: {query}"

In [0]:
class MockDoc:
    def __init__(self, content):
        self.page_content = content

class MockVectorStore:
    def similarity_search(self, query, k=5):
        return [
            MockDoc("Deep learning is a subset of machine learning using neural networks."),
            MockDoc("CNNs and transformers are popular deep learning architectures.")
        ]

class MockWebSearchTool:
    def search(self, query):
        return f"[WEB SEARCH] External info about: {query}"

class MockLLM:
    def invoke(self, prompt):
        return f"[LLM OUTPUT BASED ON CONTEXT]\n{prompt[:200]}..."

# Instantiate mocks
vectorstore = MockVectorStore()
web_search_tool = MockWebSearchTool()
llm = MockLLM()

In [0]:
corrective_rag_workflow(
    "What is deep learning?",
    vectorstore,
    web_search_tool
)

[CRAG Decision]: Insufficient internal knowledge, starting fallback retrieval.


'[LLM OUTPUT BASED ON CONTEXT]\nAnswer the question based on the following information:\n[WEB SEARCH] External info about: What is deep learning?\n\nQuestion: What is deep learning?\nAnswer:...'

In [0]:
print(set("What is deep learning?".lower().split()))
print(set("Deep learning is a subset of machine learning.".lower().split()))
print(naive_relevance_scorer(
    "What is deep learning?",
    "Deep learning is a subset of machine learning."
))

{'what', 'is', 'learning?', 'deep'}
{'machine', 'learning', 'of', 'learning.', 'subset', 'a', 'is', 'deep'}
0.5


### Adaptive RAG

工作原理：

复杂度分析：小型分类器模型对查询进行路由分发。

路径A（无需检索）： 适用于问候语或大型语言模型已掌握的常识性问题。

路径B（标准RAG）： 用于简单的事实查证。

路径C（多步智能体）： 处理需跨多源检索的复杂分析型问题。

适用场景：用户问题复杂度差异大，需要兼顾响应速度与成本。

In [0]:
from langchain_core.runnables import RunnableBranch

# 1. Define the routing function (in practice, this could be a small LLM classifier)
def route_question(query):
    simple_keywords = ["hello", "hi", "who are you"]
    complex_keywords = ["compare", "analyze", "trend", "summarize the past five years"]
    
    if any(kw in query.lower() for kw in simple_keywords):
        return "simple"
    elif any(kw in query.lower() for kw in complex_keywords):
        return "complex"
    else:
        return "standard"

# 2. Define different processing branches
def simple_chain(query):
    """Answer directly without retrieval"""
    return llm.invoke(
        f"Respond in a friendly and concise way to the user's greeting or simple question.\nQuestion: {query}"
    )

def standard_chain(query):
    """Standard RAG pipeline (reuse the existing retriever)"""
    docs = retriever.invoke(query)
    context = "\n".join([d.page_content for d in docs])
    return llm.invoke(
        f"Based on the following context:\n{context}\n\nAnswer the question: {query}"
    )

def complex_chain(query):
    """More complex workflow, e.g., multi-step retrieval or agent usage (simplified here as deeper retrieval)"""
    print("[Adaptive RAG]: Complex question detected. Enabling deep retrieval.")
    docs = retriever.invoke(query, search_kwargs={"k": 10})  # Retrieve more documents
    
    # More advanced logic could be added here (e.g., re-ranking, multi-query expansion, etc.)
    context = "\n---\n".join([d.page_content for d in docs])
    
    return llm.invoke(
        f"Please provide a comprehensive analysis based on the following information:\n{context}\n\nQuestion: {query}"
    )

# 3. Build the adaptive routing chain
branch = RunnableBranch(
    (lambda x: route_question(x["query"]) == "simple",
     lambda x: {"result": simple_chain(x["query"])}),
    
    (lambda x: route_question(x["query"]) == "complex",
     lambda x: {"result": complex_chain(x["query"])}),
    
    lambda x: {"result": standard_chain(x["query"])}  # Default branch
)

# 4. Test (Wikipedia-style)
test_queries = [
    "What is deep learning?",
    "What architectures are commonly used in deep learning?",
    "Summarize the history of deep learning and compare CNNs vs Transformers."
]

def to_text(x):
    return x.content if hasattr(x, "content") else str(x)

for q in test_queries:
    response = branch.invoke({"query": q})
    print(f"Question: {q}")
    print(f"Routing decision: {route_question(q)}")
    print(f"Answer: {to_text(response['result'])[:120]}...\n")

Question: What is deep learning?
Routing decision: standard
Answer: [LLM OUTPUT BASED ON CONTEXT]
Based on the following context:
Deep learning is a form of machine learning that transform...

Question: What architectures are commonly used in deep learning?
Routing decision: simple
Answer: [LLM OUTPUT BASED ON CONTEXT]
Respond in a friendly and concise way to the user's greeting or simple question.
Question:...

Question: Summarize the history of deep learning and compare CNNs vs Transformers.
Routing decision: simple
Answer: [LLM OUTPUT BASED ON CONTEXT]
Respond in a friendly and concise way to the user's greeting or simple question.
Question:...



### Fusion RAG

工作原理：

查询扩展：生成用户问题的3-5种变体。

并行检索：在向量数据库中搜索所有变体。

互补排序融合（RRF）：运用数学公式重新排序结果：

最终排序：在多次检索中排名靠前的文档将被提升至顶部。

适用场景：用户提问模糊、口语化，或问题本身涉及多角度时。

核心：查询扩展 + 倒数排序融合 (RRF)。